# House Prices - Advanced Regression Techniques
**Kaggle Public Score: 0.12033 | 324th place**

## Pipeline
1. Data Loading & Outlier Removal
2. Feature Engineering (numerical + categorical)
3. Target Encoding (high-cardinality categoricals)
4. Skewness Transform
5. Preprocessing (StandardScaler + OneHotEncoder)
6. Feature Selection (intersection of model importances)
7. Stacking Ensemble (LGB + XGB + GBM + CatBoost + ElasticNet)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import ElasticNet, RidgeCV
from sklearn.model_selection import cross_validate
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import category_encoders as ce

## 1. Data Loading

In [2]:
train_df = pd.read_csv('train_df.csv', keep_default_na=False)
test_df  = pd.read_csv('test_df.csv',  keep_default_na=False)

train_df = train_df.drop(columns='Id')
test_ids = test_df['Id']
test_df  = test_df.drop(columns='Id')

# MSSubClass, MoSold: 숫자형이지만 범주형으로 처리
train_df[['MSSubClass', 'MoSold']] = train_df[['MSSubClass', 'MoSold']].astype(str)
test_df[['MSSubClass',  'MoSold']] = test_df[['MSSubClass',  'MoSold']].astype(str)

## 2. Outlier Removal
상관계수/상관비 분석 결과 기반으로 극단적 이상치 제거

In [3]:
train_df = train_df[train_df['GrLivArea']   <= 4500]
train_df = train_df[train_df['LotArea']     <= 200000]
train_df = train_df[train_df['LotFrontage'] <= 300]

## 3. Feature Engineering
### 3.1 Numerical Features
상관계수 분석으로 최적 가중치를 도출한 파생 피처

In [4]:
def nums(df):
    # 거주 면적: 1층/2층 가중 합산 (상관계수 분석 기반)
    df['new_GrLivArea'] = df['1stFlrSF'] * 1.2 + df['2ndFlrSF'] * 0.7
    # 외부 공간: 포치/덱 가중 합산
    df['Porch_Deck'] = (df['OpenPorchSF'] * 1.05
                      + df['3SsnPorch']   * 0.37
                      + df['ScreenPorch'] * 0.37
                      + df['WoodDeckSF']  * 0.58)
    # 주택 연령
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    return df

train_df = nums(train_df)
test_df  = nums(test_df)

train_df = train_df.drop(columns=['YrSold', 'YearBuilt'])
test_df  = test_df.drop(columns=['YrSold', 'YearBuilt'])

### 3.2 Categorical Features
순서형 범주: 등급 → 숫자 변환  
이진 플래그: 정상/부분 거래 여부

In [5]:
# 정상/부분 거래 여부 플래그
train_df['SaleCondition_flag'] = train_df['SaleCondition'].isin(['Normal', 'Partial']).astype(int)
test_df['SaleCondition_flag']  = test_df['SaleCondition'].isin(['Normal', 'Partial']).astype(int)

# 품질 등급 → 숫자 (Ex=5, Gd=4, TA=3, Fa=2, Po=1, NA=0)
def replace_label_case1(df, col):
    grading_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    df[col] = df[col].map(grading_map).fillna(0)
    return df

# FireplaceQu: Po=0 (없음과 구분)
def replace_label_case2(df, col):
    grading_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 0, 'NA': 1}
    df[col] = df[col].map(grading_map).fillna(0)
    return df

columns_case1 = ['GarageQual', 'HeatingQC', 'PoolQC', 'ExterQual',
                 'BsmtQual', 'KitchenQual', 'GarageCond', 'ExterCond']
columns_case2 = ['BsmtCond', 'FireplaceQu']

for col in columns_case1:
    replace_label_case1(train_df, col)
    replace_label_case1(test_df, col)
for col in columns_case2:
    replace_label_case2(train_df, col)
    replace_label_case2(test_df, col)

## 4. Target & Target Encoding
- Target: log1p(SalePrice) → 왜도 정규화, RMSLE 메트릭과 일치
- Target Encoding: 고카디널리티 범주형 → 타깃 평균값으로 변환
- **No Leakage**: train 데이터로만 fit, test는 transform만 적용

In [6]:
X_train = train_df.drop(columns='SalePrice')
y_train = train_df['SalePrice']
y_log   = np.log1p(y_train)

# 카디널리티 높은 범주형 컬럼 Target Encoding
# Neighborhood(25), Exterior1st(15), Exterior2nd(16), MSSubClass(15)
target_enc_cols = ['Neighborhood', 'Exterior1st', 'Exterior2nd', 'MSSubClass']

te = ce.TargetEncoder(cols=target_enc_cols, smoothing=10)
te.fit(X_train[target_enc_cols], y_log)

X_train[target_enc_cols] = te.transform(X_train[target_enc_cols])
test_df[target_enc_cols]  = te.transform(test_df[target_enc_cols])

## 5. Skewness Transform
왜도 0.75 이상인 수치형 피처에 log1p 적용

In [7]:
num_cols    = X_train.select_dtypes(include='number').columns
skewed_cols = X_train[num_cols].apply(skew).abs()
skewed_cols = skewed_cols[skewed_cols >= 0.75].index

X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
test_df[skewed_cols]  = np.log1p(test_df[skewed_cols])

## 6. Preprocessing Pipeline
- 수치형: StandardScaler
- 범주형: OneHotEncoder

In [8]:
def num_cat_features(df):
    num_features = df.select_dtypes(include='number').columns
    cat_features = df.select_dtypes(include=['object', 'string']).columns
    return num_features, cat_features

num_features, cat_features = num_cat_features(X_train)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

X_processed       = preprocessor.fit_transform(X_train)
test_df_processed  = preprocessor.transform(test_df)
feature_names     = preprocessor.get_feature_names_out()

## 7. Feature Selection
4개 모델의 Feature Importance 교집합 → 모든 모델이 공통으로 중요하게 본 피처만 사용

In [9]:
def get_importance_features(model, feature_names):
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': model.feature_importances_
    })
    return set(importance_df[importance_df['Importance'] > 0]['Feature'])

lgb_model = lgb.LGBMRegressor(random_state=42, verbose=-1)
xgb_model = xgb.XGBRegressor(random_state=42)
gbm_model = GradientBoostingRegressor(random_state=42)
cat_model = CatBoostRegressor(random_state=42, verbose=0)

for model in [lgb_model, xgb_model, gbm_model, cat_model]:
    model.fit(X_processed, y_log)

lgb_features      = get_importance_features(lgb_model, feature_names)
xgb_features      = get_importance_features(xgb_model, feature_names)
gbm_features      = get_importance_features(gbm_model, feature_names)
catboost_features = get_importance_features(cat_model, feature_names)

inter_features = list(lgb_features & xgb_features & gbm_features & catboost_features)
print(f'선택된 피처 수: {len(inter_features)}개')

X_processed_df    = pd.DataFrame(X_processed,        columns=feature_names)
test_processed_df  = pd.DataFrame(test_df_processed,  columns=feature_names)

X_selected        = X_processed_df[inter_features]
test_df_selected  = test_processed_df[inter_features]

선택된 피처 수: 79개


## 8. Stacking Ensemble
- Base models: LGB, XGB, GBM, CatBoost (트리) + ElasticNet (선형)
- Meta model: RidgeCV
- cv=5: OOF 예측으로 Data Leakage 방지

In [10]:
lgb_best_params     = {'n_estimators': 800, 'learning_rate': 0.03624316233201206, 'max_depth': 3, 'num_leaves': 75, 'min_child_samples': 21, 'subsample': 0.8111635342652639, 'colsample_bytree': 0.6304831434719692, 'reg_alpha': 0.0010918991315994693, 'reg_lambda': 0.0051497538863298255}
xgb_best_params     = {'n_estimators': 976, 'learning_rate': 0.028194901372253063, 'max_depth': 3, 'min_child_weight': 5, 'subsample': 0.7288866619554035, 'colsample_bytree': 0.6316772621483024, 'reg_alpha': 0.13787183973684694, 'reg_lambda': 0.47062910249812745}
gbm_best_params     = {'n_estimators': 941, 'learning_rate': 0.02743211204415027, 'max_depth': 3, 'subsample': 0.6872297621992344, 'max_features': 0.4602375943728553, 'min_samples_leaf': 5}
cat_best_params     = {'iterations': 994, 'depth': 5, 'learning_rate': 0.043692556025664915, 'subsample': 0.6068969053674399, 'l2_leaf_reg': 1.3271915791457247}
elastic_best_params = {'alpha': 0.0006408925745171936, 'l1_ratio': 0.884798640263611}

best_lgb     = lgb.LGBMRegressor(**lgb_best_params, random_state=42, verbose=-1)
best_xgb     = xgb.XGBRegressor(**xgb_best_params, random_state=42)
best_gbm     = GradientBoostingRegressor(**gbm_best_params, random_state=42)
best_cat     = CatBoostRegressor(**cat_best_params, random_seed=42, verbose=0)
best_elastic = ElasticNet(**elastic_best_params, max_iter=10000)

stack_model = StackingRegressor(
    estimators=[
        ('lgb',     best_lgb),
        ('xgb',     best_xgb),
        ('gbm',     best_gbm),
        ('cat',     best_cat),
        ('elastic', best_elastic),
    ],
    final_estimator=RidgeCV(),
    cv=5,
    n_jobs=-1
)

cv_results = cross_validate(
    stack_model, X_selected, y_log,
    cv=5,
    scoring={'mse': 'neg_mean_squared_error'}
)
rmsle = np.sqrt(-cv_results['test_mse']).mean()
print(f'Stacking CV RMSLE: {rmsle:.4f}')

Stacking CV RMSLE: 0.1107


## 9. Final Prediction & Submission

In [ ]:
stack_model.fit(X_selected, y_log)

pred_stack = np.expm1(stack_model.predict(test_df_selected))

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': pred_stack})
submission.to_csv('submission_house_prices_final.csv', index=False)

print(f'Prediction summary:')
print(f"  Min   : ${submission['SalePrice'].min():,.0f}")
print(f"  Max   : ${submission['SalePrice'].max():,.0f}")
print(f"  Mean  : ${submission['SalePrice'].mean():,.0f}")
print('✅ submission_house_prices_final.csv 저장 완료')

Prediction summary:
  Min   : $46,468
  Max   : $665,989
  Mean  : $178,679
✅ submission_house_prices_final.csv 저장 완료
